In [1]:
import importlib.util
import sys
import os
import numpy as np
from SciExpeM_API.SciExpeM import SciExpeM
# path to extract_data.py and input data
sciexp_scripts_path = r"/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts"
extract_data_spec = importlib.util.spec_from_file_location(
    'extract_data', os.path.join(sciexp_scripts_path, 'extract_data.py'))
extract_data = importlib.util.module_from_spec(extract_data_spec)
sys.modules['extract_data'] = extract_data
extract_data_spec.loader.exec_module(extract_data)
print('extract_data loaded from:', extract_data.__file__)
build_sciexp_objects_spec = importlib.util.spec_from_file_location(
    'build_sciexp_objects', os.path.join(sciexp_scripts_path, 'build_sciexp_objects.py'))
build_sciexp_objects = importlib.util.module_from_spec(build_sciexp_objects_spec)
sys.modules['build_sciexp_objects'] = build_sciexp_objects
build_sciexp_objects_spec.loader.exec_module(build_sciexp_objects)
print('build_sciexp_objects loaded from:', build_sciexp_objects.__file__)

my_sciexpem = SciExpeM(username='YOUR_USERNAME',
                       password='YOUR_PASSWORD', secure=False, port=8080)
# SciExpeM(username='YOUR_USERNAME', password='YOUR_PASSWORD', secure=False, verify=False, warning=False)
# database: porta 5432 all'indirizzo 127.0.0.1
from SciExpeM_API.Models import *

CONVERT_TO_BAR = {'atm': 1.01325, 'bar': 1., 'Torr': 0.00133322, 'Pa': 1e-5}
sp_table = {}

extract_data loaded from: /Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts/extract_data.py
build_sciexp_objects loaded from: /Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts/build_sciexp_objects.py


### FilePaper
 - description*
 - reference_doi*
 - author*
 - title*
 - year*
 - volume
 - page
 - journal

In [2]:
#TEMPLATE

file_paper = FilePaper(reference_doi="10.1021/jp0683317",
                       author="Hansen, N.; Kasper, Tina, 0000-0003-3993-5316; Klippenstein, Stephen J., 0000-0001-6297-9187; Westmoreland, Philip, 0000-0002-8189-9809; Law, M. E.; Taatjes, C. A.; Kohse-HĂśinghaus, K.; Wang, J.; Cool, T. A.",
                       title="Initial Steps Of Aromatic Ring Formation In A Laminar Premixed Fuel-Rich Cyclopentene Flames",
                       year=2007,
                       description="	Hansen, N.; Kasper, Tina, 0000-0003-3993-5316; Klippenstein, Stephen J., 0000-0001-6297-9187; Westmoreland, Philip, 0000-0002-8189-9809; Law, M. E.; Taatjes, C. A.; Kohse-HĂśinghaus, K.; Wang, J.; Cool, T. A. - The Journal Of Physical Chemistry A, 2007-02-15, (111), 4081-4092",
                       )

### OpenSMOKE input file if available

In [3]:
datapath = os.path.join(sciexp_scripts_path, 'data')
osinputname = 'PF_inputOS.dic'
# WARNING LIST OF SPECIES MUST HAVE THE NAMES OF THE MECH YOU ARE GOING TO SIMULATE - OTHERWISE, NEED REPLACEMENT
inputstr, extrainfo = extract_data.process_osinput(
    datapath, osinputname, profiles=False, flameinfo=True)
# LA LISTA DI OUTPUTSPECIES DEVE AVERE SPECIE NEL MECCANISMO CHE SIMULERAI

In [4]:
print(extrainfo)

{'profileinfo': {}, 'commonprop': {'velocity': ['54.7', 'cm/s'], 'pressure': ['4.9346e-2', 'atm'], 'length': ['3', 'cm']}, 'character': {'T profile': ['0.0', '340.', '0.1', '878.', '0.2', '1443.', '0.4', '2053.5', '0.6', '2167.6', '0.8', '2135.5', '1.0', '2095.', '3.0', '1800']}, 'molefractions': [['CYC5H8', 'O2', 'AR'], ['0.16667', '0.58333', '0.25']], 'oxidizermolefractions': []}


### Common properties

- name
- units
- value
- source_type

In [5]:
# others
sourcetype = 'reported'
#######################
commonprop = []
for name, values in extrainfo['commonprop'].items():
    if name == 'velocity': # velocity is estimated from cold gas velocity and inlet T
        ci = CommonProperty(name=name, units=values[1], value=values[0], source_type='calculated')
    ci = CommonProperty(name=name, units=values[1], value=values[0], source_type=sourcetype)
    commonprop.append(ci)
    
# ADD OTHER COMMON PROPERTIES
#ci = CommonProperty(name='temperature', units='K', value='1120', source_type='reported')
#commonprop.append(ci)
############# DO NOT EDIT
# extract pressure
if 'pressure' in extrainfo['commonprop'].keys():
    Pval, Punit = extrainfo['commonprop']['pressure']
    P = float(Pval) * CONVERT_TO_BAR[Punit]


In [6]:
# ONLY PREMIXED: put manually
# # ADD RESIDENCE TIME - always calculated
rtd = 7.5e-3
units = 's'
cr = CommonProperty(name='residence time', units=units,
                    value=rtd, source_type='calculated')
commonprop.append(cr)

In [11]:
for c in commonprop:
    print(c.serialize())

{'name': 'velocity', 'units': 'cm/s', 'value': '54.7', 'source_type': 'reported'}
{'name': 'pressure', 'units': 'atm', 'value': '4.9346e-2', 'source_type': 'reported'}
{'name': 'length', 'units': 'cm', 'value': '3', 'source_type': 'reported'}
{'name': 'residence time', 'units': 's', 'value': 0.0075, 'source_type': 'calculated'}


### Initial Species

- name
- units
- value
- source_type
- configuration

In [7]:
################## PREMIXED
comp_unit = 'mole fraction'
srctype = 'reported'
config = 'premixed'
################# do not edit
inspecies, fuelobjs = build_sciexp_objects.build_initial_species(
    my_sciexpem=my_sciexpem,
    molefractions=extrainfo['molefractions'],
    source_type=srctype,
    units=comp_unit,
    configuration=config,
)

Species: ['CYC5H8', 'O2', 'AR']
Fuels: ['CYC5H8']
Mole Frac: ['0.16667', '0.58333', '0.25']
config: ['premixed', 'premixed', 'premixed']


### Data columns

- name
- label (not comuplsory)
- species_object
- units
- data
- dg_id 
- dg_label
- source_type


In [8]:
sp_table = {
    'C3H5': 'C3H5-A+C3H5-S',
    'C2H3O': 'CH3CO+CH2CHO',
    'C3H7': 'C3H7-N+C3H7-I',
	'C4H3': 'C4H3-I+C4H3-N',
	'C4H4': 'C4H4-VAC+C4H4-BTR',
	'C4H5': 'C4H5-I+C4H5-N',
	'C3H2O': 'C2HCHO+CH2CCO',
	'C4H7': 'C4H71-3+C4H71-4+IC4H7',
	'C3H6O-Al-Ac': 'CH3COCH3+C3H6O1-3+C3H6O1-2+C2H5CHO',
	'C3H6O-Pro': 'IC3H5OH+SC3H5OH+C3H5OH',
	'C4H10': 'NC4H10+IC4H10',
	'C4H4O': 'C2H3CHCO+FURAN',
	'C6H4': 'CYC6H4+LC6H4',
	'C6H8': 'C5H5CH3-2+C5H5CH3-5+MCPTD+LC6H8+CYC6H8',
	'C9H10': 'INDANE+CH3C6H42H3+C6H5C3H5-1',
	'C9H12': 'CUMENE+C6H4CH3C2H5-4+C6H4CH3C2H5-3+C6H4CH3C2H5-2+TMBENZ+C6H5C3H7',
	'C10H6': 'C10H6-12+C10H6-14',
	'C10H10': 'FC10H10+C6H5C4H5+C10H10-12+C10H10-14+C9H8CH2+C9H7CH3-3+C9H7CH3-1',
	'C9H8O': 'INDANONE+C9H7OH+C9H7OH-M+CINNAMALDEHYDE',
	'C11H10': 'C10H7CH3+C10H7CH3_106',
	'C12H10': 'ACENAPH+BIPHENYL+C10H7C2H3-1+C10H7C2H3-2',
	'C13H12': 'C6H5CH2C6H5+C10H7C3H5',
	'C14H10': 'C14H10-P+C14H10-A+C13H8CH2+C6H5CCC6H5+BIPHENYLC2H',
  # 'C14H8': 'C12H7C2H-1+C12H7C2H-5+C10H6(C2H)2_298+C10H6(C2H)2',
'C9H8':  'INDENE+C6H5C3H3+C6H5C3H3_502+C6H5CCCH3',
    'C13H10': 'FLUORENE+BENZINDENE+BENZINDENE-1HF+BENZINDENE-3HE+PHENALENE',
  #  'C10H7CH3': 'C10H7CH3+C10H7CH3_106',
}

# if not needed : empty dictionary

In [9]:
# read data
datafile = os.path.join(sciexp_scripts_path, 'data', 'PF_data.txt')
df_data = extract_data.readdata(datafile, delzero=True)
# process data
srctype = 'digitized'
label = 'experimental_data'
# x = ['temperature', 'K']
# x = ['time', 's'] # NB must be 'residence time' for concentration time profile, but only works with 'time' lol
# y = ['concentration', 'mol/cm3']
x = ['distance', 'cm']
y = ['composition', 'mole fraction']
list_exclude_species = ['C4H4', 'LC5H8', 'C6H6']  # wrong species profiles
uncert_x = []
# uncert_x = [30, 'absolute']
# uncert_y = [0.2, 'relative']
uncert_y = [0.2, 'relative'] # default uncertainty if not available in datafile

#################### DO NOT EDIT################
TINF, TSUP = extract_data.temperature_bounds(
    extrainfo,
    x=x,
    df_data=df_data,
)
print('T max and min: {} - {} K'.format(TINF, TSUP))

datacols = build_sciexp_objects.build_data_columns(
    df_data=df_data,
    my_sciexpem=my_sciexpem,
    x=x,
    y=y,
    source_type=srctype,
    label=label,
    sp_table=sp_table,
    list_exclude_species=list_exclude_species,
    uncert_x=uncert_x,
    uncert_y=uncert_y,
)

T max and min: 340.0 - 2167.6 K
AR [<Species (5)>]
H2 [<Species (1)>]
H2O [<Species (8)>]
CO [<Species (4)>]
CO2 [<Species (10)>]
O2 [<Species (2)>]
CYC5H8 [<Species (65)>]
C4H8-2 [<Species (340)>]
CYC6H10 [<Species (589)>]
CH3 [<Species (68)>]
CH2O [<Species (67)>]
CH4 [<Species (6)>]
C2H2 [<Species (51)>]
C2H3 [<Species (321)>]
C5H6 [<Species (64)>]
C7H6 [<Species (462)>]
C2H4 [<Species (53)>]
C3H4-A [<Species (15)>]
C3H5-A [<Species (330)>]
CH2CO [<Species (66)>]
C3H2 [<Species (329)>]
C3H3 [<Species (54)>]
C3H4-P [<Species (93)>]
C4H2 [<Species (11)>]
species C4H4-VAC not as preferred name: alternative names searched
C4H4-VAC [<Species (12)>]
C4H6-1 [<Species (472)>]
C4H6 [<Species (62)>]
C5H5 [<Species (63)>]
LC5H8-13 [<Species (550)>]
LC5H3-124+LC5H3-24 [<Species (544)>, <Species (545)>]
LC5H4-124 [<Species (540)>]
LC5H4-13 [<Species (541)>]
CH3CHO [<Species (69)>]
species C4H3-I not as preferred name: alternative names searched
species C4H3-N not as preferred name: alternative n

### Assemble the experiment

In [10]:
phi_inf = extract_data.equivalence_ratio(extrainfo['molefractions'])
phi_sup = phi_inf

e = Experiment(reactor='flame', # [flow reactor, flame, stirred reactor ]
               experiment_type='burner stabilized flame speciation measurement',
               file_paper=file_paper,
               #reactor_modes=['counterflow'], # da mettere in tutte le fiamme # coflow, couterflow, premixed, burner stabilized stagnation
               reactor_modes=['premixed'], # da mettere in tutte le fiamme # coflow, couterflow, premixed, burner stabilized stagnation
               data_columns=datacols, 
               initial_species=inspecies, 
               common_properties=commonprop,
               os_input_file=inputstr,
               #t_inf = 300, t_sup = 2000,
               t_inf = TINF, t_sup = TSUP,
               p_inf = P, p_sup = P,
               phi_inf = phi_inf,
               phi_sup = phi_sup,
               # fuels=fuels,
               fuels_object = fuelobjs, # devi passargli gli id delle specie che sono fuels
               comment = 'most data extracted with claude',
               #comment = 'technically not isothermal but simulated as such - T almost unvaried'
               #comment = 'C4H2 plot unclear, order might be 1e6 or 1e8 (person who digitized assumed 1e8); bath gas varies with experimental run (Ne, He), but should not affect the results'
               )

In [33]:
datacols[0].serialize()

{'name': 'distance',
 'units': 'cm',
 'data': [0.0581,
  0.107,
  0.156,
  0.206,
  0.255,
  0.304,
  0.353,
  0.411,
  0.456,
  0.505,
  0.554,
  0.604,
  0.657,
  0.702,
  0.747,
  0.805,
  0.849,
  0.899,
  0.952,
  0.997,
  1.1,
  1.19,
  1.3,
  1.41,
  1.5,
  1.6,
  1.7,
  1.8,
  1.89,
  2.0,
  2.2,
  2.39,
  2.59,
  2.79,
  2.98],
 'dg_id': 'dg1',
 'dg_label': 'experimental_data',
 'label': None,
 'source_type': 'digitized',
 'data_group_profile': None,
 'uncertainty_kind': None,
 'uncertainty_bound': None,
 'diz': None,
 'species_object': None,
 'uncertainty_reference': None}

### Send Experiment

In [34]:
my_sciexpem.insertElement(e, verbose=True)

Experiment element inserted successfully.


In [11]:
e = my_sciexpem.filterDatabase(model_name='Experiment', id = 3828)[0]

In [30]:
# aggiungi cella per verificare da sciexpem_API
e = my_sciexpem.filterDatabase(model_name='Experiment', id = 2831)[0]
print(e)
for c in e.common_properties:
    print(c.name)

<Experiment (2831)>
pressure
velocity
residence time
length
